# Load Models

In the previous notebook we saved the large model. In this notebook we have a look on how to reload it.

## Setup & Data

In [1]:
# Import packages
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import warnings

warnings.filterwarnings("ignore")

tf.keras.backend.set_floatx("float64")

# Define random seed for whole notebook
RSEED = 42

In [2]:
# Load data
df = pd.read_csv("../data/boston.csv")

# Define target
y = df.pop("MEDV")

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

In [3]:
# Scale numerical features
# Scale numerical values
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

scaler = MinMaxScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# Convert to np array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

In [4]:
!ls saved_model

my_large_model


In [5]:
!ls saved_model/my_large_model

assets
fingerprint.pb
keras_metadata.pb
saved_model.pb
variables


## SavedModel format

Reload a fresh Keras model from the saved model:

In [6]:
# Load the saved model
with tf.device("/cpu:0"):
    new_large_model = tf.keras.models.load_model("saved_model/my_large_model")

# Check its architecture
new_large_model.summary()



Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_7 (Dense)             (None, 512)               6656      
                                                                 
 dense_8 (Dense)             (None, 512)               262656    
                                                                 
 dense_9 (Dense)             (None, 512)               262656    
                                                                 
 dense_10 (Dense)            (None, 512)               262656    
                                                                 
 dense_11 (Dense)            (None, 1)                 513       
                                                                 
Total params: 795137 (6.07 MB)
Trainable params: 795137 (6.07 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [7]:
# Evaluate the restored model
with tf.device("/cpu:0"):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f"Model MSE: {mse}")


4/4 - 0s - loss: 9.7468 - mse: 138.6417 - 233ms/epoch - 58ms/step
Model MSE: 138.64173298613503


The Model MSE is higher than in the notebook before, even though we are using the same data-split. We are using the same model and the same data. Therefore, we would assume that we also receive the same MSE. 

> **Exercise:** Can you find out where this notebook varies from the procedure in the notebook before? It might help to have a look at the values in X_train in both notebooks.

<details><summary>
Click here for a hint...
</summary>
Check out how the data were preprocessed. Did both notebooks use the same scaler?
</details>

# Why does this completely ruin the model?

A neural network doesn't just learn "concepts"; it learns highly specific mathematical weights based on the exact numbers you feed it.

    StandardScaler (used in training) centers the data around 0, usually resulting in numbers between -3 and +3.

    MinMaxScaler (used in reloading) squashes all the data to be strictly between 0 and 1.

If your Neural Network spent 2,000 epochs learning how to multiply weights against numbers ranging from -3 to +3, and you suddenly start feeding it tiny numbers between 0 and 1, the math completely breaks down. The model is blindly multiplying its saved weights against the wrong scale of inputs!

    Your PM Analogy: Imagine you trained a financial forecasting AI using data measured in Euros. You save the model. Six months later, you load the model but feed it current data measured in Japanese Yen without telling it. The AI will spit out completely nonsensical predictions!

How to fix your notebook:

To get your amazing MSE score back, you simply need to change the scaler in this notebook to match the one you used during training.

In [ ]:
from sklearn.preprocessing import StandardScaler
# ...
scaler = StandardScaler()

Re-run the notebook, and your model will perfectly recreate the excellent predictions it made at the end of the previous chapter!

(Note: In real-world MLOps, engineers solve this by saving the scaler object to a file alongside the keras model, ensuring the exact same mathematical transformation is used in production!)